In [1]:
# ruff: noqa: F401
import os
import subprocess
from pathlib import Path

import board
from dotenv import load_dotenv
from mcp import StdioServerParameters, stdio_client
from rich.console import Console
from rich.markdown import Markdown
from strands import Agent, tool
from strands.models.openai import OpenAIModel
from strands.tools.mcp import MCPClient

load_dotenv(override=True)
console = Console()

### Create Agent


In [2]:
MODEL = "gpt-5.4-mini"

model = OpenAIModel(client_args={"api_key": os.getenv("OPANAI_API_KEY")}, model_id=MODEL)

agent = Agent(
    model=model, system_prompt="You are an assistant who replies in short conscise sentences. Reply in short sentences"
)

### Run the agent


In [3]:
result = await agent.invoke_async(prompt="Say Hello in Bengali")

হ্যালো

In [4]:
result.message["content"][0]["text"]

'হ্যালো'

### Using SQLite Todo board


In [5]:
board.reset_board()
board.add_goal("Read notes.txt, read its contents and convert into Bengali, and write to bengali.text")
board.show_board()

Goal #1: Read notes.txt, read its contents and convert into Bengali, and write to bengali.text

### Use tools


In [6]:
@tool
def show_todos() -> list[dict]:
    """List every tool in the todo list. A goal has a parent id is None. A step has the parent_id set to it's goal_id"""
    return board.list_todos()


@tool
def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps

    Args:
        goal_id: The id of the goal to decompose
        steps: Short description of the steps to be taken in order

    """
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}


@tool
def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (step or goal) as done and add a short summary

    Args:
        task_id: The id of the todo tak to be done
        result: A short summary of the task completed
    """
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "Done"}

In [7]:
board_agent = Agent(
    model=model,
    system_prompt=(
        "You maintain a todo board. "
        "When asked to plan, call show_todos to inspect the board. "
        "For each pending goal without steps, break it into an ordered "
        "checklist and call plan_steps to save it. "
        "Do not duplicate existing steps. "
        "Only mark tasks complete after the work has actually been performed."
    ),
    tools=[show_todos, plan_steps, complete_task],
)

result = await board_agent.invoke_async(
    prompt="Break each pending goal without steps into actionable steps and save them."
)

board.show_board()


Tool #1: show_todos

Tool #2: plan_steps
Planned the pending goal into these steps and saved them:

1. Open `notes.txt` and read its contents.
2. Translate the full contents into Bengali.
3. Write the Bengali translation to `bengali.text`.

Goal #1: Read notes.txt, read its contents and convert into Bengali, and write to bengali.text
  Step #2: Open notes.txt and read its contents.
  Step #3: Translate the full contents into Bengali.
  Step #4: Write the Bengali translation to bengali.text.

In [8]:
console.print(Markdown(result.message["content"][0]["text"]))

Planned the pending goal into these steps and saved them:                                                          

 1 Open notes.txt and read its contents.                                                                           
 2 Translate the full contents into Bengali.                                                                       
 3 Write the Bengali translation to bengali.text.

### Using MCP


In [9]:
workspace = Path("workspace").resolve()

filesystem = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
            cwd=str(workspace),
        ),
        errlog=subprocess.DEVNULL,
    )
)

In [10]:
file_agent = Agent(
    model=model,
    system_prompt="You can read and write files in your workspace> use your tools to do what is asked",
    tools=[filesystem],
)

In [11]:
result = await file_agent.invoke_async(prompt="Read notes.txt and summarize it")


Tool #1: read_text_file
The note says:

- Welcome to the team.
- You’re building a small language tutor step by step.
- Each helper should take one task from a shared board, complete it carefully, and mark it done.
- The key instruction is to read your task closely, take your time, and do your best.

In [12]:
board.show_board()

Goal #1: Read notes.txt, read its contents and convert into Bengali, and write to bengali.text
  Step #2: Open notes.txt and read its contents.
  Step #3: Translate the full contents into Bengali.
  Step #4: Write the Bengali translation to bengali.text.